# Clothes segmentation: evaluate a model and segment your own photos

Run this after a training run has produced a `best.pt`. It does not train anything.

1. `Runtime > Change runtime type`: choose **L4 GPU**. The training cells (7 and 8) do not fit a T4; cells 5 and 6 alone also work on a T4.
2. Set `RUN_NAME` in cell 1 to the run you want to look at.
3. Run the cells from top to bottom. Cell 3 asks for the new `clothseg_code.zip` once.

## 1. Settings

In [ ]:
REPO_URL = ""  # leave empty to use the uploaded zip

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"
CODE_DIR = "/content/clothes-segmentation"
CODE_VERSION = "19"
HEIGHT, WIDTH = 576, 384
SESSION_BUDGET_MIN = 600

RUN_NAME = "r1_segformer_b3"      # folder name inside <DRIVE_ROOT>/runs
CHECKPOINT = f"{DRIVE_ROOT}/runs/{RUN_NAME}/checkpoints/best.pt"

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements and prepare the dataset

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

## 5. Evaluate on the validation split

Metrics at the original image resolution, a per-class table, the confusion matrix, and the best and worst predictions.

In [ ]:
assert os.path.exists(CHECKPOINT), f"No checkpoint at {CHECKPOINT}. Check RUN_NAME, or wait until the run has finished its first epoch."
OUT = f"{DRIVE_ROOT}/reports/eval_{RUN_NAME}_val"
!python scripts/evaluate.py --checkpoint "{CHECKPOINT}" --split val --out-dir "{OUT}"

from IPython.display import Image, display
for name in ("confusion_matrix.png", "worst_examples.png", "best_examples.png"):
    display(Image(f"{OUT}/{name}"))

# The same evaluation with flip averaging: predict on the photo and on its mirror image, average the two.
import json
!python scripts/evaluate.py --checkpoint "{CHECKPOINT}" --split val --out-dir "{OUT}_flip" --flip-tta > /content/eval_flip.log 2>&1
plain, flip = (json.load(open(f"{d}/metrics.json")) for d in (OUT, f"{OUT}_flip"))
print(f"\n{'metric':<24}{'plain':>9}{'with flip':>11}{'change':>9}")
for k in ("miou", "garment_miou", "core_garment_miou", "clothing_binary_iou", "boundary_miou", "pixel_acc", "ms_per_image"):
    print(f"{k:<24}{plain[k]:>9.4f}{flip[k]:>11.4f}{flip[k] - plain[k]:>+9.4f}")

## 6. Segment your own photos

Upload one or more photos of a person. For each photo you get a label mask, a colour overlay and one transparent cut-out per garment.

In [ ]:
from google.colab import files
import glob

PHOTOS, RESULTS = "/content/my_photos", f"{DRIVE_ROOT}/reports/segment_{RUN_NAME}"
shutil.rmtree(PHOTOS, ignore_errors=True)
os.makedirs(PHOTOS)
os.chdir(PHOTOS)
uploaded = files.upload()
os.chdir(CODE_DIR)
assert uploaded, "No photo was uploaded."

!python scripts/segment.py --checkpoint "{CHECKPOINT}" --input "{PHOTOS}" --out-dir "{RESULTS}"

for path in sorted(glob.glob(f"{RESULTS}/*_overlay.jpg")):
    print(os.path.basename(path))
    display(Image(path, width=420))

## 7. Architecture experiment: SegFormer-B3 with an edge branch

Trains SegFormer-B3 with the added edge branch for 12 epochs, with the same data, seed and schedule as the baseline run `r1_segformer_b3`, then prints both side by side. About 2.5 hours. If the session ends, run cells 1 to 4 and this cell again.

In [ ]:
EDGE_RUN = "abl_segformer_b3_edge"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {EDGE_RUN} --model segformer_b3_edge --height {HEIGHT} --width {WIDTH} --epochs 12 --time-budget-min {budget}

import pandas as pd
cols = ["garment_miou", "miou", "iou_upper-clothes", "iou_pants", "iou_skirt", "iou_dress", "iou_belt", "iou_scarf", "val_loss", "train_min"]
base = pd.read_csv(f"{DRIVE_ROOT}/runs/r1_segformer_b3/val_log.csv").set_index("epoch")
assert os.path.exists(f"{DRIVE_ROOT}/runs/{EDGE_RUN}/val_log.csv"), "The edge run has not finished an epoch yet. Read the training output above for the reason."
edge = pd.read_csv(f"{DRIVE_ROOT}/runs/{EDGE_RUN}/val_log.csv").set_index("epoch")
print("\ngarment mIoU per epoch")
print(pd.DataFrame({"baseline B3": base["garment_miou"], "B3 + edge": edge["garment_miou"]}).to_string())
last = min(len(base), len(edge))
print(f"\nepoch {last}")
print(pd.DataFrame({"baseline B3": base.loc[last, cols], "B3 + edge": edge.loc[last, cols],
                    "difference": edge.loc[last, cols] - base.loc[last, cols]}).round(4).to_string())

## 8. Final training run

Trains the chosen model from the start with a longer schedule, as a new run. It takes several hours. If the session ends, run cells 1 to 4 and this cell again: it continues from the last checkpoint.

Set `FINAL_MODEL` to `"segformer_b3_edge"` only if we agreed that the edge model won the experiment above.

When it has finished, set `RUN_NAME` in cell 1 to the name printed below and run cells 5, 6 and 9 for the final model.

In [ ]:
FINAL_MODEL, FINAL_EPOCHS = "segformer_b3", 30
FINAL_RUN = f"final_{FINAL_MODEL}_e{FINAL_EPOCHS}"
print("run name:", FINAL_RUN)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert gpu_gb >= 20, (f"This GPU has {gpu_gb:.0f} GB. Training needs the L4 (about 22 GB). "
                      "Choose Runtime > Change runtime type > L4 GPU, then run cells 1 to 4 and this cell again.")

os.chdir(CODE_DIR)
budget = int(minutes_left())
!python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name {FINAL_RUN} --model {FINAL_MODEL} --height {HEIGHT} --width {WIDTH} --epochs {FINAL_EPOCHS} --time-budget-min {budget}

## 9. Final test-set evaluation

**Run this once, only for the final model.** The test split must not be used to choose between models; if it is, its numbers are no longer an honest estimate.

In [ ]:
I_CONFIRM_THIS_IS_THE_FINAL_MODEL = False

assert I_CONFIRM_THIS_IS_THE_FINAL_MODEL, "Set the flag to True only when the model and its settings are frozen."
OUT = f"{DRIVE_ROOT}/reports/eval_{RUN_NAME}_test"
!python scripts/evaluate.py --checkpoint "{CHECKPOINT}" --split test --out-dir "{OUT}"
for name in ("confusion_matrix.png", "worst_examples.png", "best_examples.png"):
    display(Image(f"{OUT}/{name}"))

## What to send back

The text output of cell 5 and a note on how the overlays in cell 6 look (which garments are right, which are wrong).